In [ ]:
!pip install pandas numpy scikit-learn nltk xgboost


In [ ]:
import pandas as pd

movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

print(movies.head())
print(credits.head())

      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en   
2  [{"id": 470, "nam

In [ ]:
movies = movies.merge(credits, on='title')

movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew',
                 'vote_average']]

movies.dropna(inplace=True)

movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",7.6
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",6.1


In [ ]:
import ast

def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)


In [ ]:
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
new_df = movies[['movie_id','title','tags','vote_average']]

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

/tmp/ipykernel_1108/3089450492.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [ ]:
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

/tmp/ipykernel_1108/3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = tfidf.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors.shape

(4806, 5000)

In [ ]:
X = vectors
y = new_df['vote_average']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor( n_estimators=100, random_state=42 ),
    "KNN": KNeighborsRegressor(),
    "XGBoost": XGBRegressor( n_estimators=100, learning_rate=0.1, random_state=42 )
    }

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
    )
import numpy as np

In [ ]:
results = []
for name, model in models.items():
  print(f"Training {name}...")
  model.fit(X_train, y_train)
  predictions = model.predict(X_test)
  mae = mean_absolute_error(y_test, predictions)
  rmse = np.sqrt(mean_squared_error(y_test, predictions))
  r2 = r2_score(y_test, predictions)
  results.append([name, mae, rmse, r2])

Training Linear Regression...
Training Decision Tree...
Training Random Forest...
Training KNN...
Training XGBoost...


In [ ]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2 Score"]
    )
results_df


,Model,MAE,RMSE,R2 Score
0,Linear Regression,1.811890,2.313917,-3.215527
1,Decision Tree,0.959875,1.374667,-0.487823
2,Random Forest,0.717344,1.056847,0.120611
3,KNN,0.846424,1.216994,-0.166093
4,XGBoost,0.739042,1.091584,0.061854


In [ ]:
final_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

final_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    recommended_movies = []

    for i in movies_list:
        recommended_movies.append(new_df.iloc[i[0]].title)

    return recommended_movies

In [ ]:
recommend("Avatar")

['Aliens',
 'Falcon Rising',
 'Battle: Los Angeles',
 'Aliens vs Predator: Requiem',
 'Apollo 18']

In [ ]:
import pickle

In [ ]:
pickle.dump(new_df, open('movies.pkl', 'wb'))

pickle.dump(similarity, open('similarity.pkl', 'wb'))

pickle.dump(final_model, open('model.pkl', 'wb'))

In [ ]:
ratings = pd.read_csv(
    '/content/u.data',
    sep='\t',
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)

ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=ratings)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
movies_ml = pd.read_csv(
    '/content/u.item',
    sep='|',
    encoding='latin-1',
    header=None
)

In [ ]:
movies_ml = movies_ml[[0, 1]]

movies_ml.columns = ['movie_id', 'title']

movies_ml.head()

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [ ]:
ratings = ratings.merge(movies_ml, on='movie_id')

ratings.head()

,user_id,movie_id,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [ ]:
avg_ratings = ratings.groupby('title')['rating'].mean().reset_index()

avg_ratings.head()

,title,rating
0,'Til There Was You (1997),2.333333
1,1-900 (1994),2.600000
2,101 Dalmatians (1996),2.908257
3,12 Angry Men (1957),4.344000
4,187 (1997),3.024390


In [ ]:
avg_ratings.rename(
    columns={'rating': 'user_rating'},
    inplace=True
)

In [ ]:
new_df = new_df.merge(
    avg_ratings,
    on='title',
    how='left'
)

In [ ]:
new_df['user_rating'] = new_df['user_rating'].fillna(
    new_df['vote_average']
)

In [ ]:
y = new_df['user_rating']

In [ ]:
X = vectors
y = new_df['user_rating']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor( n_estimators=100, random_state=42 ),
    "KNN": KNeighborsRegressor(),
    "XGBoost": XGBRegressor( n_estimators=100, learning_rate=0.1, random_state=42 )
    }

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
    )
import numpy as np

In [ ]:
results = []
for name, model in models.items():
  print(f"Training {name}...")
  model.fit(X_train, y_train)
  predictions = model.predict(X_test)
  mae = mean_absolute_error(y_test, predictions)
  rmse = np.sqrt(mean_squared_error(y_test, predictions))
  r2 = r2_score(y_test, predictions)
  results.append([name, mae, rmse, r2])

Training Linear Regression...
Training Decision Tree...
Training Random Forest...
Training KNN...
Training XGBoost...


In [ ]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2 Score"]
    )
results_df

,Model,MAE,RMSE,R2 Score
0,Linear Regression,1.811890,2.313917,-3.215527
1,Decision Tree,0.960395,1.387151,-0.514970
2,Random Forest,0.717344,1.056847,0.120611
3,KNN,0.846424,1.216994,-0.166093
4,XGBoost,0.739042,1.091584,0.061854


In [ ]:
final_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

final_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
import pickle

In [ ]:
pickle.dump(new_df, open('movies.pkl', 'wb'))

pickle.dump(similarity, open('similarity.pkl', 'wb'))

pickle.dump(final_model, open('model.pkl', 'wb'))

In [ ]:
new_df[['title', 'user_rating']].head()

,title,user_rating
0,Avatar,7.2
1,Pirates of the Caribbean: At World's End,6.9
2,Spectre,6.3
3,The Dark Knight Rises,7.6
4,John Carter,6.1


In [ ]:
pickle.dump(new_df, open('movies.pkl', 'wb'))